# Step-by-step demo of NEDAS using the vort2d model



## System setup (optional)

In Google Colab, set `to_install = True` and run this cell to install the environment.
You may need to restart the kernel after this, but you only need to run this once.

In the docker image ``myying/nedas-tutorials`` the environment is installed ready, you can skip this part.


In [ ]:
to_install = False

if to_install:
    # 1. Install the latest version of NEDAS on develop branch
    %cd ~
    %rm -rf NEDAS
    !git clone https://github.com/nansencenter/NEDAS.git
    %cd NEDAS
    %pip install .
    
    # 2. Install additional dependencies
    # numba provides JIT compilation of python function to speed it up during runtime
    %pip install numba
    # cmocean provides better colormaps for visualization
    %pip install cmocean
    # ipython widgets for interactive plots
    %pip install ipywidgets
    
    # 3. Clone the tutorial repo too
    %mkdir ~/work
    %cd ~/work
    %rm -rf NEDAS_tutorials
    !git clone https://github.com/myying/NEDAS_tutorials.git
    %cd ~/work/NEDAS_tutorials


## Initialization

A YAML file `vort2d/config.yml` contains all the settings for an experiment.

See [Configuration file](https://nedas.readthedocs.io/en/latest/config_file.html) documentation for more details.

In command line, you can run the experiment with `python -m NEDAS -c vort2d/config.yml`

Here we setup the `config` object in an interactive environment:

In [ ]:
# utility modules
import numpy as np
from datetime import datetime, timezone
import matplotlib.pyplot as plt
from IPython.display import Image, display

# key NEDAS modules
from NEDAS.config import Config
from NEDAS.schemes import get_scheme

# some utility funcs for the vort2d case
from vort2d.utils import get_time_series, get_truth, get_model_ens, get_times
from vort2d.diagnostics import rmse, sprd, grid_to_spec, variance_spec

# some graphics routines
from vort2d.graphics import plot_velocity_map, plot_vorticity_map, plot_vorticity_spaghetti
from vort2d.graphics import pwrspec2d, plot_spectrum, adjust_spec_ax, plot_ts, adjust_ts_ax
from vort2d.graphics import get_time_id_for_plot, get_hours, make_animation, animation_ui

In [ ]:
# load configuration YAML file
# additional kwargs will overwrite the values in the file
config = Config(config_file="vort2d/config.yml", nproc=1, debug=False)

# you can also change values after the initialization, like this
# config.nproc = 1
# config.debug = False

# to check all the config parameters
# config.__dict__

In [ ]:
# Once you are happy with the configuration,
# you can initialize the main Scheme object by
scheme = get_scheme(config)

## The verifying truth

We will run an Observing System Simulation Experiment (OSSE) for the vort2d model.

First step is to generate a "verifying truth", or a "nature run", which will be used both for generating synthetic observations and for verification of the results. The ``model.generate_truth`` method defines how truth run is created.

In [ ]:
%rm -rf vort2d/work/truth

# set time to the start of the experiment
scheme.c.time = config.time_start

# run the model from time_start to time_end and save results
# in offline IO mode, the truth states are saved under model.truth_dir (vort2d/work/truth/*nc files)
scheme.run_step('prepare_truth')

In [ ]:
# collect the truth states for each time step
truth_state = get_time_series(scheme.c, get_truth)

In [ ]:
# visualize the truth state over time
casename = 'truth'
plotname = 'state'

# loop over time index and make plot
hours = get_hours(scheme.c)
t_ids = get_time_id_for_plot(scheme.c)
for i, n in enumerate(t_ids):
    fig, ax = plt.subplots(1, 2, figsize=(7, 3))
    plot_velocity_map(fig, ax[0], scheme.c, hours[n], truth_state[n], vmax=30, color='k')
    plot_vorticity_map(fig, ax[1], scheme.c, hours[n], truth_state[n], colorbar=True)
    plt.savefig(f"vort2d/work/plots/{casename}_{plotname}_{i+1:02}.png")
    plt.close()

# make an animation
make_animation(scheme.c, casename, plotname)
display(Image(filename=f'vort2d/{casename}_{plotname}_animation.gif'))

## Ensemble generation

Now we generate an initial ensemble by perturbing initial conditions

The ``model.generate_init_ensemble`` method describes how the initial conditions shall be perturbed. For the vort2d model case, we randomly place the main vortex in the domain and background flow with random additive noises.

In [ ]:
%rm -rf vort2d/work/init_ens

scheme.c.time = config.time_start

# this calls the model.generate_init_ensemble and save to files (vort2d/work/init_ens/*nc)
scheme.run_step('prepare_init_ensemble')

# this prepares the first analysis cycle
scheme.run_step('preprocess')

In [ ]:
init_ens = get_model_ens(scheme.c, config.time_start)

In [ ]:
# visualize the ensemble using a spaghetti plot
fig, ax = plt.subplots(1, 1, figsize=(3,3))
hours = get_hours(scheme.c)

n = 0 # initial time step
plot_vorticity_spaghetti(fig, ax, scheme.c, hours[n], truth_state[n], init_ens)

## Data assimilation


### Assimilating a single wind observation

with large ensemble

In [ ]:
%mkdir -p vort2d/work/cycle/200101010000/analysis

config.nens = 1000

config.obs_def[0]['hroi'] = 150_000
config.obs_def[0]['nobs'] = 1
config.dataset_def['vort2d']['network_type'] = 'targeted'
config.dataset_def['vort2d']['obs_range'] = 50_000

analysis_time = datetime(2001,1,1, tzinfo=timezone.utc)
config.time = analysis_time

scheme = get_scheme(config)

scheme.run_step('prepare_init_ensemble')
scheme.run_step('preprocess')

scheme.c.iter = 0
scheme.c.update_assim_tools()

In [ ]:
from NEDAS.core import State

scheme.c.state = State(scheme.c)

scheme.c.logger('Prepare state')(scheme.c.state.prepare_state)(scheme.c)

In [ ]:
from NEDAS.core import Obs

scheme.c.obs = Obs(scheme.c)

scheme.c.logger('Prepare obs')(scheme.c.obs.prepare_obs)(scheme.c)

scheme.c.logger('Compute obs from prior state')(scheme.c.obs.prepare_obs_from_state)(scheme.c, 'prior')

In [ ]:
from NEDAS.assim_tools.assimilators import get_assimilator

scheme.c.assimilator = get_assimilator(scheme.c)

scheme.c.logger('Assimilator')(scheme.c.assimilator.assimilate)(scheme.c)

In [ ]:
from NEDAS.assim_tools.updators import get_updator
scheme.c.updator = get_updator(scheme.c)
scheme.c.logger('Updator')(scheme.c.updator.update)(scheme.c)

In [ ]:
# compute correlation
grid = scheme.c.grid
fld_prior_ens = np.zeros((config.nens,)+grid.x.shape)
fld_post_ens = np.zeros((config.nens,)+grid.x.shape)
for m in range(config.nens):
    fld_prior_ens[m,...] = scheme.c.state.fields_prior[m,0][0,...]
    fld_post_ens[m,...] = scheme.c.state.fields_post[m,0][0,...]
fld_prior_mean = np.mean(fld_prior_ens, axis=0)
fld_post_mean = np.mean(fld_post_ens, axis=0)

obs_prior_ens = np.array([scheme.c.obs.obs_prior[m,0][0] for m in range(config.nens)])
obs_prior_mean = np.mean(obs_prior_ens, axis=0)
obs_post_ens = np.array([scheme.c.obs.obs_post[m,0][0] for m in range(config.nens)])
obs_post_mean = np.mean(obs_prior_ens, axis=0)

cov = np.zeros(grid.x.shape)
fld_prior_var = np.zeros(grid.x.shape)
obs_prior_var = 0
for m in range(config.nens):
    cov += ((fld_prior_ens[m,...] - fld_prior_mean) * (obs_prior_ens[m] - obs_prior_mean))
    fld_prior_var += (fld_prior_ens[m,...] - fld_prior_mean)**2
    obs_prior_var += (obs_prior_ens[m] - obs_prior_mean)**2
cov /= config.nens-1
fld_prior_var /= config.nens-1
obs_prior_var /= config.nens-1

fld_prior_var[np.where(fld_prior_var==0)] = 1e-10

corr = cov / np.sqrt(fld_prior_var * obs_prior_var)

In [ ]:
grid = scheme.c.grid
times = get_times(scheme.c)

fig, ax = plt.subplots(1, 3, figsize=(12,4), constrained_layout=True)
state_i, state_j = 30, 30
state_x = grid.x[state_j, state_i]
state_y = grid.y[state_j, state_i]
state_prior = fld_prior_ens[:, state_j, state_i]
state_post = fld_post_ens[:, state_j, state_i]

i = times.index(analysis_time)
plot_velocity_map(ax[0], scheme.c, vmax=30)

# correlation map
cmap = getattr(cmocean.cm, 'balance') 
#grid.plot_field(ax[1], truth_state[i][0], -model.Vmax, model.Vmax, cmap)
#grid.plot_scatter(ax[1], obs_val[0], -model.Vmax, model.Vmax, 15, cmap, x=obs_x, y=obs_y)
grid.plot_field(ax[1], corr, vmin=-1, vmax=1, cmap=cmap)
#add_colorbar(fig, ax[1], cmap, -1, 1)
ax[1].plot(obs_x, obs_y, color='y', marker='o', markersize=3, zorder=10)
ax[1].plot(state_x, state_y, color='y', marker='+', markersize=10, zorder=10)
set_map_axis(ax[1], grid)

# scatter plot of obs-state relation
ax[2].scatter(obs_prior_ens, state_prior, color='c')
ax[2].scatter(obs_post_ens, state_post, color='b')


## Running DA experiments

Running the entire scheme, cycling from time_start to time_end

A few things to play with

Since the step by step shown, only works with nproc=1

If you will try nproc>1, use scheme() directly


Case

configuration

run scheme

check results, visualizations and performance diagnostics

In [ ]:
cases_output = {}

### Define cases

In [ ]:
# a global obs network with 
casename = 'assimGlobal'
config = Config(config_file='vort2d/config.yml')
# config.obs_def[0]['nobs']

In [ ]:
casename = 'assimGlobalMoreFreq'
config = Config(config_file='vort2d/config.yml', cycle_period=2)

In [ ]:
casename = 'assimGlobalLessFreq'
config = Config(config_file='vort2d/config.yml', cycle_period=12)

In [ ]:
casename = 'assimGlobalSmallEns'
config = Config(config_file='vort2d/config.yml', nens=8)
config.obs_def[0]['hroi'] = 90000.0

In [ ]:
# a different obs network, only with targeted obs near the strongest cyclonic vortex
casename = 'assimTargeted'
config = Config(config_file='vort2d/config.yml')
config.obs_def[0]['nobs'] = 300
config.dataset_def['vort2d']['network_type'] = 'targeted'
config.dataset_def['vort2d']['obs_range'] = 50_000

In [ ]:
# A free ensemble forecast without DA
casename = 'freeRun'
config = Config(config_file='vort2d/config.yml', run_analysis=False)

### Run the case and collect data

In [ ]:
scheme = get_scheme(config)

In [ ]:
# clear previous results
%rm -rf vort2d/work/cycle

In [ ]:
scheme()

In [ ]:
truth_state = get_time_series(scheme.c, get_truth)
ens_state = get_time_series(scheme.c, get_model_ens)

### Diagnostic plots

In [ ]:
plotname = 'diag'

# compute rmse and ensemble spread
rmse_ts = rmse(ens_state, truth_state)
sprd_ts = sprd(ens_state)
wn, truth_spec = grid_to_spec(truth_state)
_,  err_spec  = grid_to_spec(np.mean(ens_state, axis=1) - truth_state)
_, sprd_spec  = variance_spec(ens_state)

# save to dict for comparison later
if casename not in cases_output:
    cases_output[casename] = {
        'hours':get_hours(scheme.c),
        'rmse_ts':rmse_ts,
        'sprd_ts':sprd_ts,
        'err_spec':err_spec,
        'sprd_spec':sprd_spec,
        'wn':wn,
        'truth_spec':truth_spec,
    }

In [ ]:
%mkdir -p vort2d/work/plots

# loop over time index and make plot
hours = get_hours(scheme.c)
t_ids = get_time_id_for_plot(scheme.c)
for i, n in enumerate(t_ids):
    fig, ax = plt.subplots(1, 4, figsize=(15, 3))

    # panel 1: truth vorticity with highlighted contours in black
    plot_vorticity_map(fig, ax[0], scheme.c, hours[n], truth_state[n,...], colorbar=True)

    # panel 2: ensemble spaghetti of the highlighted contours
    plot_vorticity_spaghetti(fig, ax[1], scheme.c, hours[n], truth_state[n,...], ens_state[n,...])

    # panel 3: spectra of RMSE and ens spread compared to truth
    plot_spectrum(ax[2], wn, np.mean(truth_spec[n,...], axis=0), 'k', '-', 1, 'Truth')
    plot_spectrum(ax[2], wn, np.mean(err_spec[n,...], axis=0), 'g', '-', 3, 'Error')
    plot_spectrum(ax[2], wn, np.mean(sprd_spec[n,...], axis=0), 'r', ':', 2, 'Spread')
    ax[2].legend(loc='upper right')
    adjust_spec_ax(ax[2], scheme.c.grid.Lx, hours[n])

    # panel 4: Sawtooth time series of RMSE and ens spread
    plot_ts(ax[3], n, hours, rmse_ts, 'g', '-', 3, 'RMSE', current_marker=True)
    plot_ts(ax[3], n, hours, sprd_ts, 'r', ':', 2, 'Spread')
    ax[3].legend(loc='upper right')
    adjust_ts_ax(ax[3], hours)

    plt.savefig(f"vort2d/work/plots/{casename}_{plotname}_{i+1:02}.png")
    plt.close()

make_animation(scheme.c, casename, plotname)
print('Case: ', casename)
display(Image(filename=f'vort2d/{casename}_{plotname}_animation.gif'))

In [ ]:
# or, to view the plots in an interactive ui with a time slider
ui = animation_ui(scheme.c, casename, plotname)
display(ui)

### Compare cases

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(7,4))

cases_to_compare = ['freeRun', 'assimGlobal', 'assimGlobalLessFreq']
colors = ['k', 'g', 'c']
styles = ['-', '-', ':']
widths = [1, 2, 2]

for i,casename in enumerate(cases_to_compare):
    if casename not in cases_output:
        raise KeyError(f"Case '{casename}' not run yet!")

    # panel 1: comparison of error spectrum at last time step
    wn = cases_output[casename]['wn']
    err_spec = np.mean(cases_output[casename]['err_spec'][-1,...], axis=0)
    ax[0].loglog(wn, err_spec, color=colors[i], linestyle=styles[i], linewidth=widths[i], label=casename)

    # panel 2: comparison of error time series
    hours = cases_output[casename]['hours']
    rmse_ts = cases_output[casename]['rmse_ts']
    ax[1].plot(hours, rmse_ts, color=colors[i], linestyle=styles[i], linewidth=widths[i], label=casename)

ax[0].legend(loc='upper right')
adjust_spec_ax(ax[0], scheme.c.grid.Lx, hours[-1])
ax[1].legend(loc='upper right')
adjust_ts_ax(ax[1], hours)
